Load FAQ data

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Making structured output with `pydantic`

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

Make the instruction to generate questions

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()
openai_client = OpenAI()

user_prompt = json.dumps(doc)

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [9]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

result = response.output_parsed

print(result)

questions=['I just found this course late — can I still join and start from here?', 'Is it too late to enroll now, or is the course still open?', 'Can I join the course after it has already started?', 'If I join now, do I still have a chance to get the certificate?', 'What do I need to do to be eligible for the certificate if I’m joining late?']


In [10]:
print(result.questions)

['I just found this course late — can I still join and start from here?', 'Is it too late to enroll now, or is the course still open?', 'Can I join the course after it has already started?', 'If I join now, do I still have a chance to get the certificate?', 'What do I need to do to be eligible for the certificate if I’m joining late?']


## Using `evaluation_utils`

```py
def llm_structured(client, instructions, user_prompt, output_type, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage
```

In [11]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['I just found this course late — can I still sign up and take part?', 'Am I allowed to join the course after it has already started?', 'If I enroll now, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I join late?', 'Is it still worth joining if the submission window is already close to ending?']


## Tracking Cost

In [12]:
usage.input_tokens, usage.output_tokens

(207, 91)

In [13]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00040950000000000003,
 'total_cost': 0.00056475}

## Ground truth answer

Pair the generated questions with the correct answer (i.e. the document that should come up as search result)

In [14]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still sign up and take part?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll now, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Is it still worth joining if the submission window is already close to ending?',
  'document': '74eb249bbf'}]

Nice! We already have paired the questions and ground truth for one document, now we want to do the same with all of our documents and save them into a CSV

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

`llm_structured` makes one structured-output call. `llm_structured_retry` wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

In [15]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

Try it for the first 5 documents

In [16]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:09<00:00,  1.93s/it]


This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

## Parallel processing

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 113/113 [00:35<00:00,  3.21it/s]


One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Split the results

In [ ]:
# ground_truth = []
# usages = []

# for records, usage in results:
#     ground_truth.extend(records)
#     usages.append(usage)

# len(ground_truth)

565

Calculate the cost

In [ ]:
from evaluation_utils import calc_price

# total_cost = 0.0

# for usage in usages:
#     cost = calc_price(usage)
#     total_cost = total_cost + cost["total_cost"]

# total_cost

0.08658149999999999

In [ ]:
from evaluation_utils import calc_total_price

# calc_total_price(usages)

0.08658149999999999

In [ ]:
import pandas as pd

# df_ground_truth = pd.DataFrame(ground_truth)
# df_ground_truth

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I have to submit my project before submissi...,74eb249bbf
4,What do I need to do to receive the course cer...,74eb249bbf
...,...,...
560,Why is pip giving me requests 2.28 when I need...,4b30b918bc
561,What’s the command to install requests straigh...,4b30b918bc
562,If I’m seeing a 401 Client Error while using t...,4b30b918bc
563,How do I fix the requests version issue withou...,4b30b918bc


Save to CSV

In [ ]:
# df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

Try loading up the document and convert it into a dictionary

In [27]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit my project before submissions close to get certified?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to receive the course certificate if I join late?',
  'document': '74eb249bbf'}]

In [28]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [29]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [30]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

In [31]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [32]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [33]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

Wrap it in a function

In [34]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

Sample check

In [35]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Can I still join the course if I found it late?


[1, 0, 0, 0, 0]

Put it in a loop to run through all generated questions and their ground truth answers

In [36]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [37]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

100%|██████████| 15/15 [00:00<00:00, 53.60it/s]


In [38]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0]]

We can also make the compute functions more flexible by replacing `text_search` with `search_function` and make `search_function` as a parameter. This way, we can change `search_function` with any search function we want to use (e.g. vector search, hybrid search)

In [39]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

If we put `text_search` as the `search_function`, we would get the exact same result as we got above

In [40]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

100%|██████████| 15/15 [00:00<00:00, 309.50it/s]


[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0]]

Now let's run it for the whole ground truth questions

In [42]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total[:5]

100%|██████████| 565/565 [00:01<00:00, 299.15it/s]


[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

# Search Evaluation Metrics

## Hit Rate

If a line contains 1, search found the correct document somewhere in the top 5 results. If the line contains only zeros, search did not find the correct document.

For example, out of 100 ground truth, 90 of them have 1 in it, 10 of them only have zeros. In that case, the hit rate is 90/100 = 0.9 (90%).

In [43]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

## Mean Reciprocal Rank (MRR)

For each query, the score is based on the rank of the first correct document:

- position 1: score is 1.0
- position 2: score is 0.5
- position 3: score is 0.333
- not found: score is 0

In [44]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

## Combine both metrics

In [45]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [46]:
evaluate(
    ground_truth,
    text_search
)

100%|██████████| 565/565 [00:01<00:00, 292.78it/s]


{'hit_rate': 0.8300884955752212, 'mrr': 0.7111799410029497}

# Search Parameter Tuning

Let's make a search function which the value of `boost_dict` for `"question"` can be varied

In [47]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [48]:
# Let's try several boost values
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

100%|██████████| 565/565 [00:02<00:00, 274.37it/s]


boost=0.5: {'hit_rate': 0.8920353982300885, 'mrr': 0.789380530973451}


100%|██████████| 565/565 [00:02<00:00, 273.32it/s]


boost=1.0: {'hit_rate': 0.8867256637168142, 'mrr': 0.7755457227138638}


100%|██████████| 565/565 [00:01<00:00, 291.60it/s]


boost=3.0: {'hit_rate': 0.8300884955752212, 'mrr': 0.7111799410029497}


100%|██████████| 565/565 [00:01<00:00, 283.91it/s]


boost=5.0: {'hit_rate': 0.8035398230088495, 'mrr': 0.6773746312684364}


100%|██████████| 565/565 [00:01<00:00, 310.84it/s]

boost=10.0: {'hit_rate': 0.7663716814159292, 'mrr': 0.6564306784660766}


We can also tune `answer` and `section` together with `question`.

In [49]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [50]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


100%|██████████| 565/565 [00:02<00:00, 279.63it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


100%|██████████| 565/565 [00:01<00:00, 298.09it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

100%|██████████| 565/565 [00:01<00:00, 291.53it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


100%|██████████| 565/565 [00:01<00:00, 311.84it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


100%|██████████| 565/565 [00:01<00:00, 335.70it/s]


Sort by MRR:

In [51]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.976991,0.863068
6,1.0,4.0,0.1,0.973451,0.861593
23,2.0,10.0,0.5,0.976991,0.860826
22,2.0,10.0,0.2,0.973451,0.859115
21,2.0,10.0,0.1,0.971681,0.858466
4,1.0,2.0,0.2,0.966372,0.856726
8,1.0,4.0,0.5,0.973451,0.855634
3,1.0,2.0,0.1,0.966372,0.855605
35,5.0,10.0,0.5,0.966372,0.855605
19,2.0,4.0,0.2,0.966372,0.855605


The best result has
- `question = 1`
- `answer = 4`
- `section = 0.2`

Let's modify our `text_search` function and implement the best boosts

In [52]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 4.0,
        "section": 0.2,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

# Evaluating Answers

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course if you discovered it late.\n\nIf you want a certificate, you need to submit your project while the course is still accepting submissions.'

In [11]:
assistant.total_cost()

0.0005505000000000001

In [12]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [13]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you discovered it late.\n\nIf you want a certificate, you need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [15]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. But if you want a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [16]:
assistant.reset_usage()

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, ground_truth, generate_rag_answer)

100%|██████████| 565/565 [02:16<00:00,  4.15it/s]


In [ ]:
# answers = []

# for answer_record in results:
#     answers.append(answer_record)

In [20]:
assistant.total_cost()

0.6068437499999999

In [ ]:
# df_answers = pd.DataFrame(answers)
# df_answers.to_csv("data/rag-answers-new.csv", index=False)

## LLM Judge

In [26]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [23]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [24]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [25]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [27]:
rec = answers[0]

In [28]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [29]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key points of the ground truth: late joining is allowed, but certificate eligibility requires submitting the project while submissions are still being accepted. This is semantically equivalent.', score='good')

In [30]:
calc_price(usage)

{'input_cost': 0.00022574999999999998,
 'output_cost': 0.0002385,
 'total_cost': 0.00046425}

In [31]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [32]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the exact meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project while submissions are still being accepted. No key information is missing or altered.', score='good')

In [33]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [34]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

100%|██████████| 565/565 [02:29<00:00,  3.79it/s]


In [35]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [36]:
df_eval = pd.DataFrame(evaluations)

In [37]:
calc_total_price(usages)

0.39483599999999996

In [38]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 540/565 = 95.58%


In [39]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Is it too late to start this course now?,74eb249bbf,bad,The ground truth says it is not too late to st...
28,Do I need to be in the live class to do the re...,69d122f12e,bad,The AI answer fails to answer the question and...
124,"I already added the key, but notebook still ca...",86d99bbf21,bad,The AI answer does not provide the required so...
162,What should I install and put in my shell conf...,8b2f5e9d04,bad,The ground truth asks specifically for a shell...
187,What do I need installed if I want to work thr...,aa310de435,bad,The AI answer does not convey the ground truth...


# Agent Evaluation

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## Using ToyAIKit to create a runner

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I found it late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join course after start found it late add drop self-paced late registration"}', call_id='call_7r6UOU2HZBpZbwAnDIqNGyTv', name='search', type='function_call', id='fc_0c16eed169dacfbf006a54f22f0690819aab56443f58ce296a', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_7r6UOU2HZBpZbwAnDIqNGyTv',
  'output': '[\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a cer